In [ ]:
# 2026-10-20
import pandas as pd
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from apis_core.apis_metainfo.models import Collection, Uri
from apis_core.apis_entities.models import Person
from apis_core.apis_vocabularies.models import PersonPlaceRelation

In [ ]:
col = Collection.objects.get(id=84)
domain = "burgtheater"
born_in_rel = PersonPlaceRelation.objects.get(name="geboren in")
died_in_rel = PersonPlaceRelation.objects.get(name="gestorben in")
born_in_rel, died_in_rel, col, domain
uri_stub = "https://kulturerbe.burgtheater.at/person/"


In [ ]:
df = gsheet_to_df("1Q_E8vaackJHEJeAuNxQgbGgHW7B3TKyuRB-lcaHIZkI")

In [ ]:
df.head(3)

In [ ]:
# add bt-uris to existing entities
for i, row in tqdm(df.iterrows(), total=len(df)):
    if pd.isna(row["pmb"]):
        continue
    else:
        pmb_id = int(row["pmb"])
    person = Person.objects.get(id=pmb_id)
    person.collection.add(col)
    bt_id = f"{uri_stub}{row['burgtheater_id']}"
    bt_uri, _ = Uri.objects.get_or_create(uri=bt_id, entity=person, domain=domain)

In [ ]:
# add bt-uris to existing entities
for i, row in tqdm(df.iterrows(), total=len(df)):
    if pd.isna(row["pmb"]):
        bt_id = f"{uri_stub}{row['burgtheater_id']}"
        pmb_uri, _ = Uri.objects.get_or_create(uri=bt_id, domain=domain)
        entity = pmb_uri.entity
        if entity:
            person = Person.objects.get(id=entity.id)
            pass
        else:
            item = {
                "name": row["last_name"][:254] if pd.notna(row["last_name"]) else "",
                "first_name": row["first_name"][:254] if pd.notna(row["first_name"]) else "",
            }
            entity = Person.objects.create(**item)
            person = Person.objects.get(id=entity.id)
            pmb_uri.entity = entity
            pmb_uri.save()
        entity.collection.add(col)